# MLOps: Model Optimization for Production

---

**Course Module:** ML Foundation — MLOps Track  
**Focus:** Taking trained models from research notebooks to production-ready deployments  

---

## Table of Contents

1. [Why Model Optimization Matters](#1-why-model-optimization-matters)
2. [Model Compression Techniques](#2-model-compression-techniques)  
   2.1 Quantization (INT8, FP16)  
   2.2 Pruning  
   2.3 Knowledge Distillation  
   2.4 ONNX Format for Cross-Platform Deployment  
3. [Model Serving Strategies](#3-model-serving-strategies)  
   3.1 Batch vs. Real-Time Inference  
   3.2 Edge vs. Cloud Deployment  
   3.3 Latency and Throughput Considerations  
4. [Hands-On Project: Optimize a CV Model for Deployment](#4-hands-on-project)  
   4.1 Train a Baseline CV Model  
   4.2 Convert to ONNX Format  
   4.3 Apply Quantization  
   4.4 Profile Performance  
   4.5 Compare Optimized vs. Original  
5. [CV Project Development Sprint](#5-cv-project-development-sprint)  
   5.1 Inference Pipeline  
   5.2 Performance Documentation  
   5.3 Visualizations and Demo  
   5.4 Comprehensive Project Report  
6. [Summary and Next Steps](#6-summary-and-next-steps)

---

### Dataset Used in This Notebook

We use **CIFAR-10**, a widely-used benchmark dataset containing 60,000 32×32 color images across 10 classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck). It ships natively with PyTorch/torchvision, making it ideal for a self-contained tutorial. The techniques demonstrated here transfer directly to larger models and datasets in production.

**Alternative datasets you could substitute:**
- **ImageNet (subset)** — for more realistic image classification at scale  
- **COCO** — for object detection / segmentation tasks  
- **Oxford-IIIT Pets** — a fine-grained classification dataset  
- **Fashion-MNIST** — a lightweight drop-in replacement for quick experiments  

---

## 0. Environment Setup

Before we begin, install and import all required libraries. This notebook is designed to run on **CPU** so everyone can follow along — GPU availability simply speeds up training.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch torchvision onnx onnxruntime psutil matplotlib numpy tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.quantization
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms

import onnx
import onnxruntime as ort

import numpy as np
import matplotlib.pyplot as plt
import time
import os
import psutil
import copy
from pathlib import Path

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")
print(f"ONNX Runtime version: {ort.__version__}")

---

# 1. Why Model Optimization Matters <a id='1-why-model-optimization-matters'></a>

When a deep learning model works well in a Jupyter notebook, the natural next question is: *"How do we get this into the hands of users?"* The jump from research to production reveals three critical bottlenecks:

### 1.1 Latency

**Latency** is the time between sending an input to the model and receiving a prediction. In research, waiting 500ms per image is fine. In production:

| Application | Required Latency | Why |
|---|---|---|
| Self-driving cars | < 10ms | Safety-critical real-time decisions |
| Web search ranking | < 50ms | Users abandon slow results |
| Mobile photo filter | < 100ms | Feels "instant" to the user |
| Batch document processing | < 1s/doc | Throughput matters more than single-request speed |

A model that runs at 200ms on a V100 GPU may run at 2000ms on a mobile CPU. Optimization bridges this gap.

### 1.2 Memory

A standard ResNet-50 uses ~100MB of weights in FP32. That's manageable on a server, but consider:

- **Mobile phones** have 3–8GB total RAM shared across all apps
- **Edge devices** (Raspberry Pi, Jetson Nano) may have 1–4GB
- **Serverless functions** (AWS Lambda) have strict memory caps (128MB–10GB)
- **Serving multiple models** on one GPU means memory must be shared

Reducing model size from 100MB to 25MB isn't just a nice-to-have — it can determine whether deployment is even *possible*.

### 1.3 Compute Cost

Cloud GPU instances are expensive. At scale, inference cost often exceeds training cost:

- Training a model once: **one-time cost**
- Serving predictions to 1M users/day: **ongoing, compounding cost**

A 2× speedup in inference directly halves your GPU bill. For companies serving billions of requests, even a 10% improvement translates to millions in annual savings.

### The Optimization Landscape

```
                        Model Optimization
                              │
         ┌────────────────────┼────────────────────┐
         │                    │                     │
    Compression          Conversion            Serving
         │                    │                     │
  ┌──────┼──────┐        ONNX/TensorRT      ┌─────┼─────┐
  │      │      │                            │     │     │
Quant  Prune  Distill                     Batch  Real  Edge
                                          Inf.   Time  Deploy
```

In this notebook, we will work through each branch of this tree with real code.

Let's make this concrete. We'll measure the baseline cost of running a standard model so we can appreciate the impact of optimizations later.

In [ ]:
def get_model_size_mb(model):
    """Calculate model size in megabytes from its parameters."""
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    total_bytes = param_size + buffer_size
    return total_bytes / (1024 ** 2)


def get_file_size_mb(filepath):
    """Get the size of a saved file in megabytes."""
    return os.path.getsize(filepath) / (1024 ** 2)


def count_parameters(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def measure_inference_latency(model, input_tensor, n_runs=100, warmup=10):
    """
    Measure average inference latency over n_runs.
    Includes a warmup phase to stabilize measurements.
    """
    model.eval()
    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input_tensor)

    # Timed runs
    latencies = []
    with torch.no_grad():
        for _ in range(n_runs):
            start = time.perf_counter()
            _ = model(input_tensor)
            end = time.perf_counter()
            latencies.append((end - start) * 1000)  # ms

    return {
        "mean_ms": np.mean(latencies),
        "std_ms": np.std(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
    }


# Quick demonstration: measure a pretrained ResNet-18
resnet18 = torchvision.models.resnet18(weights=None)
dummy_input = torch.randn(1, 3, 32, 32)

total_params, trainable_params = count_parameters(resnet18)
model_mb = get_model_size_mb(resnet18)
latency = measure_inference_latency(resnet18, dummy_input, n_runs=50)

print(f"ResNet-18 Baseline Profile")
print(f"{'─' * 40}")
print(f"Total parameters:    {total_params:>12,}")
print(f"Model size (FP32):   {model_mb:>11.2f} MB")
print(f"Mean latency:        {latency['mean_ms']:>11.2f} ms")
print(f"P95 latency:         {latency['p95_ms']:>11.2f} ms")
print(f"\nThis is our starting point. Let's make it smaller and faster.")

---

# 2. Model Compression Techniques <a id='2-model-compression-techniques'></a>

Model compression is the art of making models smaller and faster **without significantly sacrificing accuracy**. There are four primary techniques, each attacking a different source of redundancy.

---

## 2.1 Quantization: INT8 and FP16

### The Core Idea

Neural network weights are normally stored as **32-bit floating-point numbers** (FP32). Quantization reduces the precision of these numbers:

| Format | Bits | Range | Use Case |
|---|---|---|---|
| FP32 | 32 | ±3.4×10³⁸ | Training (default) |
| FP16 | 16 | ±65,504 | GPU inference, mixed precision training |
| INT8 | 8 | -128 to 127 | CPU/edge inference |
| INT4 | 4 | -8 to 7 | Aggressive compression (LLMs) |

**Why does this work?** Trained neural network weights tend to cluster in narrow ranges. The full dynamic range of FP32 is wasted. By mapping weights to a smaller data type, we:

1. **Reduce memory** — 4× smaller with INT8 vs FP32
2. **Speed up computation** — integer arithmetic is faster than floating-point on most hardware
3. **Reduce energy** — fewer bits moved through memory buses

### Types of Quantization

- **Post-Training Static Quantization (PTQ):** Calibrate quantization parameters using a representative dataset *after* training. No retraining needed.
- **Post-Training Dynamic Quantization:** Weights are quantized ahead of time; activations are quantized on-the-fly during inference. Simplest to apply.
- **Quantization-Aware Training (QAT):** Insert fake quantization operations during training so the model *learns* to be robust to reduced precision. Best accuracy, most effort.

In this section we'll demonstrate **dynamic quantization** (simplest) and **static quantization** (most common in production).

In [ ]:
# ──────────────────────────────────────────────
# Step 1: Build and train a small CNN on CIFAR-10
# ──────────────────────────────────────────────
# We define a compact CNN so training completes quickly on CPU.
# The optimization techniques apply identically to larger models.

class CIFAR10CNN(nn.Module):
    """
    A compact CNN for CIFAR-10 classification.
    Architecture: 3 conv blocks → adaptive pool → 2 FC layers.
    ~270K parameters — small enough to train on CPU in minutes.
    """

    def __init__(self, num_classes=10):
        super().__init__()
        # --- Feature extractor ---
        self.features = nn.Sequential(
            # Block 1: 3 → 32 channels
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 2: 32 → 64 channels
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 3: 64 → 128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        # --- Classifier ---
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


# Print architecture summary
model = CIFAR10CNN().to(DEVICE)
total, trainable = count_parameters(model)
print(f"CIFAR10CNN Architecture")
print(f"{'─' * 40}")
print(f"Total parameters:  {total:>10,}")
print(f"Model size (FP32): {get_model_size_mb(model):.2f} MB")
print(f"\n{model}")

In [ ]:
# ──────────────────────────────────────────────
# Step 2: Load CIFAR-10 dataset
# ──────────────────────────────────────────────

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

# Download CIFAR-10
train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)
test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

# For faster iteration, use a subset for training (full test set for evaluation)
train_subset = Subset(train_dataset, range(10000))  # 10K samples
calib_subset = Subset(test_dataset, range(1000))     # 1K for quantization calibration

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)
calib_loader = DataLoader(calib_subset, batch_size=64, shuffle=False, num_workers=0)

CLASSES = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"Training samples:    {len(train_subset):>6,}")
print(f"Test samples:        {len(test_dataset):>6,}")
print(f"Calibration samples: {len(calib_subset):>6,}")
print(f"Classes:             {len(CLASSES)}")

In [ ]:
# ──────────────────────────────────────────────
# Step 3: Visualize some samples
# ──────────────────────────────────────────────

def imshow_grid(dataset, n=8, title="CIFAR-10 Samples"):
    """Display a grid of images from the dataset."""
    fig, axes = plt.subplots(1, n, figsize=(14, 2))
    mean = np.array([0.4914, 0.4822, 0.4465])
    std = np.array([0.2470, 0.2435, 0.2616])

    for i in range(n):
        img, label = dataset[i]
        img = img.numpy().transpose(1, 2, 0)
        img = img * std + mean  # unnormalize
        img = np.clip(img, 0, 1)
        axes[i].imshow(img)
        axes[i].set_title(CLASSES[label], fontsize=9)
        axes[i].axis("off")

    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

imshow_grid(test_dataset)

In [ ]:
# ──────────────────────────────────────────────
# Step 4: Train the baseline model
# ──────────────────────────────────────────────

def train_model(model, train_loader, test_loader, epochs=10, lr=0.001):
    """Train a model and return training history."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "test_acc": []}

    for epoch in range(epochs):
        # ---- Training ----
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        scheduler.step()

        # ---- Evaluation ----
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        test_acc = 100.0 * correct / total

        history["train_loss"].append(avg_loss)
        history["test_acc"].append(test_acc)

        print(f"Epoch {epoch+1:>2}/{epochs} │ "
              f"Loss: {avg_loss:.4f} │ "
              f"Test Acc: {test_acc:.2f}%")

    return history


def evaluate_accuracy(model, test_loader, device=DEVICE):
    """Evaluate model accuracy on a test set."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100.0 * correct / total


# Train
print("Training baseline CIFAR10CNN...")
print("=" * 50)
history = train_model(model, train_loader, test_loader, epochs=10, lr=0.001)

baseline_acc = history["test_acc"][-1]
print(f"\n✓ Baseline accuracy: {baseline_acc:.2f}%")

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], "o-", color="#e74c3c", linewidth=2, markersize=5)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training Loss")
ax1.set_title("Training Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(history["test_acc"], "o-", color="#2ecc71", linewidth=2, markersize=5)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Test Accuracy (%)")
ax2.set_title("Test Accuracy")
ax2.grid(True, alpha=0.3)

plt.suptitle("Baseline Model Training", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Save baseline model
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/baseline_cifar10.pth")
baseline_file_size = get_file_size_mb("models/baseline_cifar10.pth")
print(f"Baseline model saved: {baseline_file_size:.2f} MB")

### Dynamic Quantization (INT8)

Dynamic quantization is the simplest form — a single function call. It quantizes **weights** to INT8 at save time and quantizes **activations** dynamically during inference. It's most effective on layers dominated by matrix multiplication (Linear layers, LSTMs).

In [ ]:
# ──────────────────────────────────────────────
# Dynamic Quantization
# ──────────────────────────────────────────────

# Move model to CPU (quantization is CPU-only in PyTorch)
model_cpu = copy.deepcopy(model).cpu()

# Apply dynamic quantization — targets Linear layers
model_dynamic_quant = torch.quantization.quantize_dynamic(
    model_cpu,
    qconfig_spec={nn.Linear},  # Quantize Linear layers
    dtype=torch.qint8,         # Target INT8
)

# Save and measure
torch.save(model_dynamic_quant.state_dict(), "models/dynamic_quant_cifar10.pth")
dq_file_size = get_file_size_mb("models/dynamic_quant_cifar10.pth")

# Evaluate accuracy
dq_acc = evaluate_accuracy(model_dynamic_quant, test_loader, device="cpu")

print(f"Dynamic Quantization Results")
print(f"{'─' * 45}")
print(f"Original size:    {baseline_file_size:.2f} MB")
print(f"Quantized size:   {dq_file_size:.2f} MB")
print(f"Compression:      {baseline_file_size / dq_file_size:.2f}×")
print(f"Original acc:     {baseline_acc:.2f}%")
print(f"Quantized acc:    {dq_acc:.2f}%")
print(f"Accuracy drop:    {baseline_acc - dq_acc:.2f}%")

### Static Quantization (INT8)

Static quantization goes further: it quantizes both **weights and activations** ahead of time. This requires a **calibration step** — we run a representative dataset through the model so it can learn the typical range of activation values. The result is faster inference than dynamic quantization because activations don't need to be quantized on-the-fly.

Static quantization requires modifications to the model architecture: we need to insert `QuantStub` and `DeQuantStub` modules to mark where tensors should enter and exit the quantized domain.

In [ ]:
# ──────────────────────────────────────────────
# Static Quantization
# ──────────────────────────────────────────────

class QuantizableCIFAR10CNN(nn.Module):
    """
    Same architecture as CIFAR10CNN, but with QuantStub/DeQuantStub
    for static quantization support.
    """

    def __init__(self, num_classes=10):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.dequant = torch.quantization.DeQuantStub()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.quant(x)        # Enter quantized domain
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        x = self.dequant(x)      # Exit quantized domain
        return x


# Create quantizable model and load trained weights
quant_model = QuantizableCIFAR10CNN()
# Load weights from baseline (architectures match except for quant stubs)
quant_model.features.load_state_dict(model_cpu.features.state_dict())
quant_model.classifier.load_state_dict(model_cpu.classifier.state_dict())
quant_model.eval()

# Step 1: Fuse layers (Conv+BN+ReLU → single optimized op)
# This is critical for both speed and quantization accuracy.
quant_model_fused = torch.quantization.fuse_modules(
    quant_model,
    [
        ["features.0", "features.1", "features.2"],  # Conv1+BN+ReLU
        ["features.4", "features.5", "features.6"],  # Conv2+BN+ReLU
        ["features.8", "features.9", "features.10"], # Conv3+BN+ReLU
    ],
)

# Step 2: Specify quantization config
quant_model_fused.qconfig = torch.quantization.get_default_qconfig("x86")

# Step 3: Prepare for calibration
torch.quantization.prepare(quant_model_fused, inplace=True)

# Step 4: Calibrate with representative data
print("Calibrating with representative data...")
with torch.no_grad():
    for images, _ in calib_loader:
        quant_model_fused(images)

# Step 5: Convert to quantized model
torch.quantization.convert(quant_model_fused, inplace=True)

# Save and evaluate
torch.save(quant_model_fused.state_dict(), "models/static_quant_cifar10.pth")
sq_file_size = get_file_size_mb("models/static_quant_cifar10.pth")
sq_acc = evaluate_accuracy(quant_model_fused, test_loader, device="cpu")

print(f"\nStatic Quantization Results")
print(f"{'─' * 45}")
print(f"Original size:    {baseline_file_size:.2f} MB")
print(f"Quantized size:   {sq_file_size:.2f} MB")
print(f"Compression:      {baseline_file_size / sq_file_size:.2f}×")
print(f"Original acc:     {baseline_acc:.2f}%")
print(f"Quantized acc:    {sq_acc:.2f}%")
print(f"Accuracy drop:    {baseline_acc - sq_acc:.2f}%")

### FP16 (Half Precision)

FP16 is even simpler to apply than INT8 and gives a guaranteed 2× size reduction. It's especially effective on GPUs with Tensor Cores (NVIDIA Volta and newer). On CPU, the speed benefit is less pronounced, but the memory saving is still significant.

In [ ]:
# ──────────────────────────────────────────────
# FP16 Half Precision
# ──────────────────────────────────────────────

model_fp16 = copy.deepcopy(model).cpu().half()  # Convert all params to FP16

# Save and measure
torch.save(model_fp16.state_dict(), "models/fp16_cifar10.pth")
fp16_file_size = get_file_size_mb("models/fp16_cifar10.pth")

# Evaluate (need FP16 inputs)
model_fp16.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model_fp16(images.half())  # Input must also be FP16
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
fp16_acc = 100.0 * correct / total

print(f"FP16 Half Precision Results")
print(f"{'─' * 45}")
print(f"Original size:    {baseline_file_size:.2f} MB")
print(f"FP16 size:        {fp16_file_size:.2f} MB")
print(f"Compression:      {baseline_file_size / fp16_file_size:.2f}×")
print(f"Original acc:     {baseline_acc:.2f}%")
print(f"FP16 acc:         {fp16_acc:.2f}%")
print(f"Accuracy drop:    {baseline_acc - fp16_acc:.2f}%")

---

## 2.2 Pruning: Removing Unnecessary Connections

### The Core Idea

Research consistently shows that neural networks are **over-parameterized** — many weights contribute little to the output. Pruning identifies and removes these weights, creating a **sparse** model.

Think of it like sculpting: you start with a block of marble (a fully-connected model) and chisel away everything that isn't the statue (useful weights).

### Types of Pruning

- **Unstructured pruning:** Zero out individual weights (fine-grained, but needs sparse hardware to accelerate)
- **Structured pruning:** Remove entire filters, channels, or layers (coarser, but runs faster on standard hardware because it removes actual computation)

### The Lottery Ticket Hypothesis

A landmark 2019 paper by Frankle & Carlin showed that within a randomly initialized network, there exists a subnetwork (the "winning ticket") that can match the full network's accuracy when trained in isolation. This means up to 90% of weights may be unnecessary — a profound result that motivates aggressive pruning.

In [ ]:
# ──────────────────────────────────────────────
# Unstructured Pruning with PyTorch
# ──────────────────────────────────────────────

import torch.nn.utils.prune as prune

def apply_global_pruning(model, amount=0.3):
    """
    Apply global unstructured L1 pruning to all Conv2d and Linear layers.
    `amount` is the fraction of weights to prune (0.3 = 30%).
    """
    parameters_to_prune = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            parameters_to_prune.append((module, "weight"))

    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )
    return model


def get_sparsity(model):
    """Calculate the percentage of zero-valued weights."""
    total_zeros = 0
    total_elements = 0
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            if hasattr(module, 'weight'):
                total_zeros += torch.sum(module.weight == 0).item()
                total_elements += module.weight.nelement()
    return 100.0 * total_zeros / total_elements if total_elements > 0 else 0


# Test different pruning levels
pruning_results = []

for prune_pct in [0.0, 0.2, 0.4, 0.6, 0.8, 0.9]:
    # Fresh copy each time
    pruned_model = copy.deepcopy(model).cpu()

    if prune_pct > 0:
        pruned_model = apply_global_pruning(pruned_model, amount=prune_pct)

    # Evaluate
    acc = evaluate_accuracy(pruned_model, test_loader, device="cpu")
    sparsity = get_sparsity(pruned_model)

    pruning_results.append({
        "prune_pct": prune_pct,
        "sparsity": sparsity,
        "accuracy": acc,
    })
    print(f"Pruning {prune_pct*100:>5.1f}% │ Sparsity: {sparsity:>5.1f}% │ Acc: {acc:.2f}%")

print("\nKey insight: notice how accuracy holds up until aggressive pruning levels!")

In [ ]:
# Visualize pruning vs accuracy tradeoff
prune_pcts = [r["prune_pct"] * 100 for r in pruning_results]
accs = [r["accuracy"] for r in pruning_results]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(prune_pcts, accs, "o-", color="#3498db", linewidth=2.5, markersize=8)
ax.axhline(y=baseline_acc, color="#e74c3c", linestyle="--", label=f"Baseline ({baseline_acc:.1f}%)")
ax.fill_between(prune_pcts, [a - 1 for a in accs], [a + 1 for a in accs],
                alpha=0.15, color="#3498db")
ax.set_xlabel("Weights Pruned (%)")
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("The Pruning-Accuracy Tradeoff", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 2.3 Knowledge Distillation: Student-Teacher Models

### The Core Idea

Knowledge distillation trains a **small model** (the student) to mimic a **large model** (the teacher). Instead of training the student on hard labels ("this is a cat"), we train it on the teacher's **soft probability distribution** ("70% cat, 15% dog, 8% tiger...").

Why are soft labels better? They contain **dark knowledge** — information about the relationships between classes. When a teacher says an image is "70% cat, 15% dog", the student learns that cats and dogs look more similar to each other than cats and trucks. This inter-class relationship information is lost with hard labels.

### The Distillation Loss

The training loss combines two objectives:

$$\mathcal{L} = \alpha \cdot \mathcal{L}_{\text{hard}}(y_{\text{student}}, y_{\text{true}}) + (1 - \alpha) \cdot T^2 \cdot \text{KL}\big(\sigma(z_s / T) \| \sigma(z_t / T)\big)$$

Where:
- $\alpha$ controls the balance between hard and soft targets
- $T$ is the **temperature** — higher values produce softer probability distributions  
- $z_s, z_t$ are the student and teacher logits  
- $\sigma$ is the softmax function  

The temperature parameter is the clever part: at T=1 (normal softmax), the teacher's output is "peaky" (one class dominates). At T=5 or T=10, the distribution flattens out, revealing more nuanced relationships between classes.

In [ ]:
# ──────────────────────────────────────────────
# Knowledge Distillation: Teacher → Student
# ──────────────────────────────────────────────

class StudentCNN(nn.Module):
    """
    A much smaller student model (~28K params vs ~270K in the teacher).
    This is intentionally underpowered — distillation should help it
    outperform training from scratch.
    """

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 4 * 4, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


student = StudentCNN().to(DEVICE)
student_params, _ = count_parameters(student)
teacher_params, _ = count_parameters(model)

print(f"Teacher parameters: {teacher_params:>10,}")
print(f"Student parameters: {student_params:>10,}")
print(f"Compression ratio:  {teacher_params / student_params:.1f}×")

In [ ]:
# ──────────────────────────────────────────────
# Distillation training loop
# ──────────────────────────────────────────────

def distill(
    teacher, student, train_loader, test_loader,
    epochs=10, lr=0.001, temperature=5.0, alpha=0.3
):
    """
    Train a student model using knowledge distillation from the teacher.

    Args:
        temperature: Softens the teacher's probability distribution.
                     Higher T → softer → more dark knowledge revealed.
        alpha:       Weight for the hard-label loss. (1-alpha) for soft loss.
    """
    teacher.eval()  # Teacher is frozen
    optimizer = optim.Adam(student.parameters(), lr=lr)
    criterion_hard = nn.CrossEntropyLoss()
    criterion_soft = nn.KLDivLoss(reduction="batchmean")

    history = {"train_loss": [], "test_acc": []}

    for epoch in range(epochs):
        student.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            # Forward pass
            student_logits = student(images)
            with torch.no_grad():
                teacher_logits = teacher(images)

            # Hard loss: student vs true labels
            loss_hard = criterion_hard(student_logits, labels)

            # Soft loss: student vs teacher (with temperature)
            loss_soft = criterion_soft(
                F.log_softmax(student_logits / temperature, dim=1),
                F.softmax(teacher_logits / temperature, dim=1),
            )

            # Combined loss (T² scaling compensates for softmax gradient magnitude)
            loss = alpha * loss_hard + (1 - alpha) * (temperature ** 2) * loss_soft

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        test_acc = evaluate_accuracy(student, test_loader)
        history["train_loss"].append(avg_loss)
        history["test_acc"].append(test_acc)

        print(f"Epoch {epoch+1:>2}/{epochs} │ Loss: {avg_loss:.4f} │ Acc: {test_acc:.2f}%")

    return history


# Also train the same student WITHOUT distillation for comparison
print("═" * 55)
print("Training student WITHOUT distillation (baseline)")
print("═" * 55)
student_scratch = StudentCNN().to(DEVICE)
scratch_history = train_model(student_scratch, train_loader, test_loader, epochs=10)

print(f"\n{'═' * 55}")
print("Training student WITH distillation (T=5, α=0.3)")
print("═" * 55)
student_distilled = StudentCNN().to(DEVICE)
distill_history = distill(
    teacher=model,
    student=student_distilled,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=10,
    temperature=5.0,
    alpha=0.3,
)

In [ ]:
# Compare distilled vs scratch student
fig, ax = plt.subplots(figsize=(8, 5))
epochs_range = range(1, 11)

ax.plot(epochs_range, scratch_history["test_acc"], "o-",
        color="#e67e22", linewidth=2, label="Student (trained from scratch)")
ax.plot(epochs_range, distill_history["test_acc"], "s-",
        color="#9b59b6", linewidth=2, label="Student (distilled from teacher)")
ax.axhline(y=baseline_acc, color="#2ecc71", linestyle="--",
           label=f"Teacher ({baseline_acc:.1f}%)")

ax.set_xlabel("Epoch")
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Knowledge Distillation: Student vs. Scratch Training", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

scratch_final = scratch_history["test_acc"][-1]
distill_final = distill_history["test_acc"][-1]
print(f"\nStudent (scratch):    {scratch_final:.2f}%")
print(f"Student (distilled):  {distill_final:.2f}%")
print(f"Distillation gain:    +{distill_final - scratch_final:.2f}%")
print(f"Student size:         {get_model_size_mb(student_distilled):.2f} MB ")
print(f"Teacher size:         {get_model_size_mb(model):.2f} MB")

---

## 2.4 ONNX Format for Cross-Platform Deployment

### What is ONNX?

**ONNX** (Open Neural Network Exchange) is an open standard for representing machine learning models. It solves a fundamental production problem: your model was trained in PyTorch, but your deployment target might be:

- A C++ application using TensorRT
- A mobile app using Core ML (iOS) or TFLite (Android)
- A browser using ONNX.js or WebAssembly
- A server using ONNX Runtime (Microsoft's optimized engine)

ONNX acts as a **universal intermediate format**. You export once, deploy everywhere:

```
PyTorch Model  ──export──>  ONNX  ──convert──>  TensorRT (NVIDIA GPU)
                                  ──convert──>  Core ML  (iPhone/Mac)
                                  ──convert──>  TFLite   (Android)
                                  ──run────>  ONNX Runtime (any platform)
```

### Why ONNX Runtime is Fast

ONNX Runtime applies **graph-level optimizations** that PyTorch's eager execution cannot:

1. **Operator fusion:** Combines consecutive operations (e.g., Conv → BN → ReLU becomes a single op)
2. **Constant folding:** Pre-computes operations that don't depend on input
3. **Memory planning:** Optimizes tensor allocation to minimize copies
4. **Hardware-specific kernels:** Selects the fastest implementation for your CPU/GPU

In [ ]:
# ──────────────────────────────────────────────
# Export PyTorch model to ONNX
# ──────────────────────────────────────────────

model_cpu = copy.deepcopy(model).cpu()
model_cpu.eval()

# Dummy input defines the expected input shape
dummy_input = torch.randn(1, 3, 32, 32)

onnx_path = "models/cifar10_model.onnx"

torch.onnx.export(
    model_cpu,
    dummy_input,
    onnx_path,
    export_params=True,            # Store trained weights inside the model
    opset_version=13,              # ONNX operator set version
    do_constant_folding=True,      # Optimize constant expressions
    input_names=["input"],         # Name the input tensor
    output_names=["output"],       # Name the output tensor
    dynamic_axes={                 # Allow variable batch size
        "input": {0: "batch_size"},
        "output": {0: "batch_size"},
    },
)

# Validate the exported model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

onnx_file_size = get_file_size_mb(onnx_path)
print(f"ONNX Export Successful!")
print(f"{'─' * 40}")
print(f"ONNX file size:  {onnx_file_size:.2f} MB")
print(f"PyTorch size:    {baseline_file_size:.2f} MB")
print(f"ONNX opset:      13")
print(f"Dynamic batch:   Yes")

In [ ]:
# ──────────────────────────────────────────────
# Run inference with ONNX Runtime
# ──────────────────────────────────────────────

# Create an inference session
ort_session = ort.InferenceSession(onnx_path)

# Verify input/output names and shapes
print("ONNX Runtime Session Info:")
for inp in ort_session.get_inputs():
    print(f"  Input:  {inp.name:>10s}  shape={inp.shape}  type={inp.type}")
for out in ort_session.get_outputs():
    print(f"  Output: {out.name:>10s}  shape={out.shape}  type={out.type}")

# Run a single prediction
sample_input = dummy_input.numpy()
ort_result = ort_session.run(None, {"input": sample_input})
print(f"\nSample prediction shape: {ort_result[0].shape}")
print(f"Predicted class: {CLASSES[np.argmax(ort_result[0])]}")

In [ ]:
# ──────────────────────────────────────────────
# Validate ONNX model accuracy matches PyTorch
# ──────────────────────────────────────────────

def evaluate_onnx_accuracy(ort_session, test_loader):
    """Evaluate ONNX model accuracy on the full test set."""
    correct, total = 0, 0
    input_name = ort_session.get_inputs()[0].name

    for images, labels in test_loader:
        ort_outputs = ort_session.run(None, {input_name: images.numpy()})
        predictions = np.argmax(ort_outputs[0], axis=1)
        correct += (predictions == labels.numpy()).sum()
        total += labels.size(0)

    return 100.0 * correct / total


onnx_acc = evaluate_onnx_accuracy(ort_session, test_loader)
print(f"PyTorch accuracy: {baseline_acc:.2f}%")
print(f"ONNX accuracy:    {onnx_acc:.2f}%")
print(f"Difference:       {abs(baseline_acc - onnx_acc):.4f}%")
print(f"\n✓ Accuracy is preserved through ONNX conversion.")

---

# 3. Model Serving Strategies <a id='3-model-serving-strategies'></a>

Having an optimized model is only half the story. The **serving strategy** — how you deliver predictions to users — is equally important.

---

## 3.1 Batch vs. Real-Time Inference

### Real-Time (Online) Inference

The model processes one request at a time (or micro-batches) with strict latency requirements.

**Best for:** User-facing applications — search ranking, recommendation, chatbots, real-time image classification.

**Characteristics:**
- Latency-sensitive (< 50–200ms)
- Unpredictable traffic patterns (need auto-scaling)
- Model must be always loaded in memory
- Cost: pay for idle capacity during low traffic

### Batch (Offline) Inference

The model processes large volumes of data at once, typically on a schedule.

**Best for:** ETL pipelines, daily content moderation, bulk document classification, pre-computing recommendations.

**Characteristics:**
- Throughput-optimized (maximize total items/second)
- Latency is less critical (minutes to hours acceptable)
- Can use larger batch sizes for GPU efficiency
- Cost: spin up, process, shut down — pay only for what you use

| Factor | Real-Time | Batch |
|---|---|---|
| Latency target | < 100ms | Minutes to hours |
| Batch size | 1–8 | 64–512+ |
| Scaling | Horizontal auto-scale | Vertical (bigger instance) |
| Freshness | Instant | Periodic |
| Cost model | Always-on (expensive) | On-demand (efficient) |

---

## 3.2 Edge Deployment vs. Cloud Deployment

### Cloud Deployment

Model runs on powerful remote servers. The client sends data over the network and receives predictions.

**Advantages:** Access to powerful GPUs, easy to update models, centralized monitoring.  
**Disadvantages:** Network latency, bandwidth costs, privacy concerns (data leaves device), requires connectivity.

### Edge Deployment

Model runs directly on the user's device (phone, IoT sensor, browser, embedded system).

**Advantages:** Zero network latency, works offline, data stays on device (privacy), no server cost.  
**Disadvantages:** Limited compute/memory, harder to update, must support diverse hardware.

### Decision Framework

```
Does the model need to run offline?  ─── Yes ──→  EDGE
       │ No
Is data privacy critical?  ─── Yes ──→  EDGE
       │ No
Is the model very large (>500MB)?  ─── Yes ──→  CLOUD
       │ No
Is sub-10ms latency required?  ─── Yes ──→  EDGE
       │ No
Either works. Choose based on cost and operational preference.
```

---

## 3.3 Latency and Throughput Considerations

**Latency** and **throughput** are often at odds:

- **Smaller batches** → lower latency (each request processed quickly), lower throughput (GPU underutilized)
- **Larger batches** → higher latency (requests wait to fill a batch), higher throughput (GPU fully utilized)

Production systems use **adaptive batching**: collect requests in a short window (e.g., 10ms), batch whatever has arrived, and process together. This balances both goals.

Key metrics to monitor:

- **P50 latency:** Median response time
- **P95/P99 latency:** Tail latency — the experience of your slowest 5%/1% of users
- **Throughput:** Requests per second (RPS) the system can sustain
- **GPU utilization:** Percentage of compute being used (aim for > 70%)

In [ ]:
# ──────────────────────────────────────────────
# Demonstrate batch size impact on throughput and latency
# ──────────────────────────────────────────────

def benchmark_batch_sizes(model, batch_sizes, input_shape=(3, 32, 32), n_runs=30):
    """
    Measure latency and throughput across different batch sizes.
    This simulates the real-time vs. batch inference tradeoff.
    """
    model.eval()
    results = []

    for bs in batch_sizes:
        dummy = torch.randn(bs, *input_shape)

        # Warmup
        with torch.no_grad():
            for _ in range(5):
                _ = model(dummy)

        # Timed runs
        latencies = []
        with torch.no_grad():
            for _ in range(n_runs):
                start = time.perf_counter()
                _ = model(dummy)
                end = time.perf_counter()
                latencies.append((end - start) * 1000)

        mean_lat = np.mean(latencies)
        throughput = (bs / mean_lat) * 1000  # images per second

        results.append({
            "batch_size": bs,
            "mean_latency_ms": mean_lat,
            "throughput_ips": throughput,
            "per_image_ms": mean_lat / bs,
        })

    return results


batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
batch_results = benchmark_batch_sizes(model_cpu, batch_sizes)

print(f"{'Batch':>6} │ {'Latency (ms)':>13} │ {'Throughput':>13} │ {'Per Image':>11}")
print(f"{'─' * 55}")
for r in batch_results:
    print(f"{r['batch_size']:>6} │ {r['mean_latency_ms']:>10.2f} ms │ "
          f"{r['throughput_ips']:>9.0f} img/s │ {r['per_image_ms']:>8.2f} ms")

In [ ]:
# Visualize the latency-throughput tradeoff
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

bs_vals = [r["batch_size"] for r in batch_results]
lat_vals = [r["mean_latency_ms"] for r in batch_results]
tp_vals = [r["throughput_ips"] for r in batch_results]
pi_vals = [r["per_image_ms"] for r in batch_results]

ax1.plot(bs_vals, lat_vals, "o-", color="#e74c3c", linewidth=2, label="Total Latency")
ax1_twin = ax1.twinx()
ax1_twin.plot(bs_vals, pi_vals, "s--", color="#3498db", linewidth=2, label="Per-Image Latency")
ax1.set_xlabel("Batch Size")
ax1.set_ylabel("Total Latency (ms)", color="#e74c3c")
ax1_twin.set_ylabel("Per-Image Latency (ms)", color="#3498db")
ax1.set_title("Latency vs. Batch Size")
ax1.set_xscale("log", base=2)
ax1.grid(True, alpha=0.3)

ax2.bar(range(len(bs_vals)), tp_vals, color="#2ecc71", alpha=0.8)
ax2.set_xticks(range(len(bs_vals)))
ax2.set_xticklabels(bs_vals)
ax2.set_xlabel("Batch Size")
ax2.set_ylabel("Throughput (images/sec)")
ax2.set_title("Throughput vs. Batch Size")
ax2.grid(True, alpha=0.3, axis="y")

plt.suptitle("The Batch Size Tradeoff", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nKey takeaway: larger batches improve throughput (amortize overhead)")
print("but increase total latency per request.")

---

# 4. Hands-On Project: Optimize a CV Model for Deployment <a id='4-hands-on-project'></a>

Now we bring everything together. We'll take our trained CIFAR-10 model through the full optimization pipeline:

1. **Baseline** → 2. **ONNX conversion** → 3. **ONNX quantization** → 4. **Profile all variants** → 5. **Compare**

---

## 4.1 ONNX Quantization

ONNX Runtime has its own quantization tools that work directly on ONNX models — you don't need to go back to PyTorch.

In [ ]:
# ──────────────────────────────────────────────
# ONNX Runtime Quantization (INT8)
# ──────────────────────────────────────────────

from onnxruntime.quantization import quantize_dynamic, QuantType

onnx_quant_path = "models/cifar10_model_int8.onnx"

# Dynamic quantization of the ONNX model
quantize_dynamic(
    model_input=onnx_path,
    model_output=onnx_quant_path,
    weight_type=QuantType.QUInt8,  # Quantize weights to UINT8
)

onnx_quant_size = get_file_size_mb(onnx_quant_path)
print(f"ONNX Quantization Complete")
print(f"{'─' * 40}")
print(f"Original ONNX:   {onnx_file_size:.2f} MB")
print(f"Quantized ONNX:  {onnx_quant_size:.2f} MB")
print(f"Compression:     {onnx_file_size / onnx_quant_size:.2f}×")

In [ ]:
# Validate quantized ONNX accuracy
ort_quant_session = ort.InferenceSession(onnx_quant_path)
onnx_quant_acc = evaluate_onnx_accuracy(ort_quant_session, test_loader)

print(f"Quantized ONNX accuracy: {onnx_quant_acc:.2f}%")
print(f"Accuracy drop from baseline: {baseline_acc - onnx_quant_acc:.2f}%")

## 4.2 Comprehensive Performance Profiling

Now let's profile every variant we've created: PyTorch baseline, quantized PyTorch, ONNX, quantized ONNX, FP16, and the distilled student. We'll measure latency, memory, throughput, and accuracy.

In [ ]:
# ──────────────────────────────────────────────
# Comprehensive profiling suite
# ──────────────────────────────────────────────

def measure_onnx_latency(session, input_shape=(1, 3, 32, 32), n_runs=100, warmup=10):
    """Measure ONNX Runtime inference latency."""
    input_name = session.get_inputs()[0].name
    dummy = np.random.randn(*input_shape).astype(np.float32)

    # Warmup
    for _ in range(warmup):
        session.run(None, {input_name: dummy})

    latencies = []
    for _ in range(n_runs):
        start = time.perf_counter()
        session.run(None, {input_name: dummy})
        end = time.perf_counter()
        latencies.append((end - start) * 1000)

    return {
        "mean_ms": np.mean(latencies),
        "std_ms": np.std(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
    }


def measure_throughput(model_or_session, is_onnx=False, batch_size=32, n_batches=20):
    """Measure throughput in images per second."""
    if is_onnx:
        input_name = model_or_session.get_inputs()[0].name
        dummy = np.random.randn(batch_size, 3, 32, 32).astype(np.float32)
        # warmup
        for _ in range(3):
            model_or_session.run(None, {input_name: dummy})
        start = time.perf_counter()
        for _ in range(n_batches):
            model_or_session.run(None, {input_name: dummy})
        elapsed = time.perf_counter() - start
    else:
        model_or_session.eval()
        dummy = torch.randn(batch_size, 3, 32, 32)
        # warmup
        with torch.no_grad():
            for _ in range(3):
                model_or_session(dummy)
        start = time.perf_counter()
        with torch.no_grad():
            for _ in range(n_batches):
                model_or_session(dummy)
        elapsed = time.perf_counter() - start

    total_images = batch_size * n_batches
    return total_images / elapsed


# ---- Profile all model variants ----
dummy_input_cpu = torch.randn(1, 3, 32, 32)

profiles = {}

# 1. PyTorch Baseline (FP32)
lat = measure_inference_latency(model_cpu, dummy_input_cpu)
tp = measure_throughput(model_cpu)
profiles["PyTorch FP32"] = {
    "accuracy": baseline_acc,
    "size_mb": baseline_file_size,
    "latency_ms": lat["mean_ms"],
    "p95_ms": lat["p95_ms"],
    "throughput_ips": tp,
}

# 2. PyTorch Dynamic Quantized (INT8)
lat = measure_inference_latency(model_dynamic_quant, dummy_input_cpu)
tp = measure_throughput(model_dynamic_quant)
profiles["PyTorch INT8 (Dynamic)"] = {
    "accuracy": dq_acc,
    "size_mb": dq_file_size,
    "latency_ms": lat["mean_ms"],
    "p95_ms": lat["p95_ms"],
    "throughput_ips": tp,
}

# 3. PyTorch FP16
lat = measure_inference_latency(model_fp16, dummy_input_cpu.half())
tp = measure_throughput(model_fp16)  # uses default fp32 input; OK for throughput estimate
profiles["PyTorch FP16"] = {
    "accuracy": fp16_acc,
    "size_mb": fp16_file_size,
    "latency_ms": lat["mean_ms"],
    "p95_ms": lat["p95_ms"],
    "throughput_ips": tp,
}

# 4. ONNX Runtime (FP32)
lat = measure_onnx_latency(ort_session)
tp = measure_throughput(ort_session, is_onnx=True)
profiles["ONNX Runtime FP32"] = {
    "accuracy": onnx_acc,
    "size_mb": onnx_file_size,
    "latency_ms": lat["mean_ms"],
    "p95_ms": lat["p95_ms"],
    "throughput_ips": tp,
}

# 5. ONNX Runtime Quantized (INT8)
lat = measure_onnx_latency(ort_quant_session)
tp = measure_throughput(ort_quant_session, is_onnx=True)
profiles["ONNX Runtime INT8"] = {
    "accuracy": onnx_quant_acc,
    "size_mb": onnx_quant_size,
    "latency_ms": lat["mean_ms"],
    "p95_ms": lat["p95_ms"],
    "throughput_ips": tp,
}

# 6. Distilled Student
student_cpu = copy.deepcopy(student_distilled).cpu()
lat = measure_inference_latency(student_cpu, dummy_input_cpu)
tp = measure_throughput(student_cpu)
profiles["Distilled Student"] = {
    "accuracy": distill_final,
    "size_mb": get_model_size_mb(student_cpu),
    "latency_ms": lat["mean_ms"],
    "p95_ms": lat["p95_ms"],
    "throughput_ips": tp,
}

# ---- Print results table ----
print(f"\n{'Model Variant':<25} │ {'Acc (%)':>8} │ {'Size (MB)':>9} │ "
      f"{'Latency':>9} │ {'P95 (ms)':>9} │ {'Throughput':>11}")
print(f"{'═' * 90}")
for name, p in profiles.items():
    print(f"{name:<25} │ {p['accuracy']:>7.2f}% │ {p['size_mb']:>8.2f}  │ "
          f"{p['latency_ms']:>7.2f}ms │ {p['p95_ms']:>7.2f}ms │ "
          f"{p['throughput_ips']:>8.0f} img/s")

In [ ]:
# ──────────────────────────────────────────────
# Visual comparison of all optimization strategies
# ──────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
names = list(profiles.keys())
colors = ["#e74c3c", "#e67e22", "#f39c12", "#2ecc71", "#3498db", "#9b59b6"]
x_pos = range(len(names))

# Accuracy
accs = [profiles[n]["accuracy"] for n in names]
axes[0, 0].barh(x_pos, accs, color=colors, alpha=0.85)
axes[0, 0].set_yticks(x_pos)
axes[0, 0].set_yticklabels(names, fontsize=9)
axes[0, 0].set_xlabel("Accuracy (%)")
axes[0, 0].set_title("Accuracy", fontweight="bold")
for i, v in enumerate(accs):
    axes[0, 0].text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=8)

# Model Size
sizes = [profiles[n]["size_mb"] for n in names]
axes[0, 1].barh(x_pos, sizes, color=colors, alpha=0.85)
axes[0, 1].set_yticks(x_pos)
axes[0, 1].set_yticklabels(names, fontsize=9)
axes[0, 1].set_xlabel("Size (MB)")
axes[0, 1].set_title("Model Size", fontweight="bold")
for i, v in enumerate(sizes):
    axes[0, 1].text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=8)

# Latency
lats = [profiles[n]["latency_ms"] for n in names]
axes[1, 0].barh(x_pos, lats, color=colors, alpha=0.85)
axes[1, 0].set_yticks(x_pos)
axes[1, 0].set_yticklabels(names, fontsize=9)
axes[1, 0].set_xlabel("Mean Latency (ms)")
axes[1, 0].set_title("Inference Latency", fontweight="bold")
for i, v in enumerate(lats):
    axes[1, 0].text(v + 0.05, i, f"{v:.2f}ms", va="center", fontsize=8)

# Throughput
tps = [profiles[n]["throughput_ips"] for n in names]
axes[1, 1].barh(x_pos, tps, color=colors, alpha=0.85)
axes[1, 1].set_yticks(x_pos)
axes[1, 1].set_yticklabels(names, fontsize=9)
axes[1, 1].set_xlabel("Throughput (images/sec)")
axes[1, 1].set_title("Throughput", fontweight="bold")
for i, v in enumerate(tps):
    axes[1, 1].text(v + 10, i, f"{v:.0f}", va="center", fontsize=8)

plt.suptitle("Model Optimization: Comprehensive Comparison",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---

# 5. CV Project Development Sprint <a id='5-cv-project-development-sprint'></a>

Now we'll put everything into a production-ready package: build a complete inference pipeline, document performance, create demo visualizations, and write a project report.

---

## 5.1 Production Inference Pipeline

A production inference pipeline wraps the raw model in a clean API with preprocessing, postprocessing, error handling, and logging.

In [ ]:
# ──────────────────────────────────────────────
# Production-ready ONNX Inference Pipeline
# ──────────────────────────────────────────────

class CIFAR10Pipeline:
    """
    A production-ready inference pipeline for CIFAR-10 classification.
    Uses ONNX Runtime for cross-platform deployment.

    Features:
    - Preprocessing built-in (normalization, resizing)
    - Batch inference support
    - Confidence thresholding
    - Top-K predictions
    - Performance logging
    """

    CLASS_NAMES = (
        'airplane', 'automobile', 'bird', 'cat', 'deer',
        'dog', 'frog', 'horse', 'ship', 'truck'
    )

    # CIFAR-10 normalization constants
    MEAN = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32)
    STD = np.array([0.2470, 0.2435, 0.2616], dtype=np.float32)

    def __init__(self, model_path, confidence_threshold=0.5):
        """
        Initialize the pipeline with an ONNX model.

        Args:
            model_path: Path to .onnx file
            confidence_threshold: Minimum confidence to return a prediction
        """
        self.session = ort.InferenceSession(model_path)
        self.input_name = self.session.get_inputs()[0].name
        self.confidence_threshold = confidence_threshold
        self.inference_count = 0
        self.total_latency_ms = 0.0
        print(f"Pipeline loaded: {model_path}")
        print(f"Confidence threshold: {confidence_threshold}")

    def preprocess(self, images_np):
        """
        Preprocess numpy images (H, W, C) in [0, 255] to model input format.
        Handles both single images and batches.
        """
        if images_np.ndim == 3:
            images_np = images_np[np.newaxis, ...]  # Add batch dim

        # Normalize to [0, 1]
        x = images_np.astype(np.float32) / 255.0

        # Apply CIFAR-10 normalization
        x = (x - self.MEAN) / self.STD

        # HWC → CHW (NCHW format)
        x = x.transpose(0, 3, 1, 2)
        return x

    def predict(self, images_np, top_k=3):
        """
        Run inference and return structured predictions.

        Args:
            images_np: Numpy array of shape (H, W, C) or (N, H, W, C)
            top_k: Number of top predictions to return

        Returns:
            List of prediction dicts with class, confidence, and top_k results
        """
        # Preprocess
        input_tensor = self.preprocess(images_np)

        # Inference with timing
        start = time.perf_counter()
        outputs = self.session.run(None, {self.input_name: input_tensor})
        latency_ms = (time.perf_counter() - start) * 1000

        # Update stats
        self.inference_count += input_tensor.shape[0]
        self.total_latency_ms += latency_ms

        # Postprocess: softmax → structured results
        logits = outputs[0]
        probabilities = self._softmax(logits)

        results = []
        for i in range(probabilities.shape[0]):
            probs = probabilities[i]
            top_indices = np.argsort(probs)[::-1][:top_k]

            prediction = {
                "class": self.CLASS_NAMES[top_indices[0]],
                "confidence": float(probs[top_indices[0]]),
                "above_threshold": float(probs[top_indices[0]]) >= self.confidence_threshold,
                "top_k": [
                    {"class": self.CLASS_NAMES[idx], "confidence": float(probs[idx])}
                    for idx in top_indices
                ],
                "latency_ms": latency_ms / input_tensor.shape[0],
            }
            results.append(prediction)

        return results

    def get_stats(self):
        """Return pipeline performance statistics."""
        avg_latency = (self.total_latency_ms / self.inference_count
                       if self.inference_count > 0 else 0)
        return {
            "total_inferences": self.inference_count,
            "avg_latency_ms": avg_latency,
            "total_time_ms": self.total_latency_ms,
        }

    @staticmethod
    def _softmax(x):
        e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return e_x / e_x.sum(axis=1, keepdims=True)


# ---- Demo the pipeline ----
pipeline = CIFAR10Pipeline(
    model_path=onnx_quant_path,  # Use the optimized INT8 model
    confidence_threshold=0.5,
)

In [ ]:
# ──────────────────────────────────────────────
# Run pipeline on test images
# ──────────────────────────────────────────────

# Get raw (unnormalized) test images
raw_test = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=False,
    transform=transforms.ToTensor()  # Just [0,1], no normalization
)

# Pick 8 diverse samples
sample_indices = [0, 100, 200, 300, 400, 500, 600, 700]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for idx, ax in zip(sample_indices, axes.flat):
    img_tensor, true_label = raw_test[idx]
    # Convert to HWC numpy for pipeline
    img_np = (img_tensor.numpy().transpose(1, 2, 0) * 255).astype(np.uint8)

    result = pipeline.predict(img_np, top_k=3)[0]

    # Display
    ax.imshow(img_np)
    color = "green" if result["class"] == CLASSES[true_label] else "red"
    ax.set_title(
        f"True: {CLASSES[true_label]}\n"
        f"Pred: {result['class']} ({result['confidence']:.1%})",
        color=color, fontsize=9,
    )
    ax.axis("off")

plt.suptitle("Inference Pipeline Demo (INT8 Quantized ONNX)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

stats = pipeline.get_stats()
print(f"\nPipeline Stats:")
print(f"  Total inferences: {stats['total_inferences']}")
print(f"  Avg latency:      {stats['avg_latency_ms']:.2f} ms/image")

## 5.2 Performance Documentation

Every production model deployment should include a **Model Card** — a document that records the model's intended use, performance characteristics, limitations, and optimization details.

In [ ]:
# ──────────────────────────────────────────────
# Generate Model Card
# ──────────────────────────────────────────────

def generate_model_card(profiles, best_model_name):
    """Generate a comprehensive model card."""
    best = profiles[best_model_name]
    baseline = profiles["PyTorch FP32"]

    card = f"""
╔══════════════════════════════════════════════════════════════╗
║                       MODEL CARD                            ║
╠══════════════════════════════════════════════════════════════╣
║ Model Name:    CIFAR-10 Image Classifier (Optimized)        ║
║ Task:          10-class image classification                 ║
║ Architecture:  Custom CNN (3 conv blocks + 2 FC layers)      ║
║ Dataset:       CIFAR-10 (50K train / 10K test)               ║
║ Framework:     PyTorch → ONNX Runtime                        ║
╠══════════════════════════════════════════════════════════════╣
║ OPTIMIZATION SUMMARY                                         ║
╠══════════════════════════════════════════════════════════════╣
║ Recommended variant:  {best_model_name:<38s} ║
║ Accuracy:             {best['accuracy']:.2f}%{' ' * 33}║
║ Model size:           {best['size_mb']:.2f} MB (was {baseline['size_mb']:.2f} MB){' ' * 18}║
║ Mean latency:         {best['latency_ms']:.2f} ms{' ' * 31}║
║ Throughput:           {best['throughput_ips']:.0f} images/sec{' ' * 24}║
╠══════════════════════════════════════════════════════════════╣
║ OPTIMIZATION TECHNIQUES APPLIED                              ║
╠══════════════════════════════════════════════════════════════╣
║ ✓ ONNX conversion (graph optimization, operator fusion)      ║
║ ✓ INT8 dynamic quantization (weights quantized to uint8)     ║
║ ✓ Constant folding enabled during export                     ║
║ ✓ Dynamic batching support (variable batch size)             ║
╠══════════════════════════════════════════════════════════════╣
║ LIMITATIONS                                                  ║
╠══════════════════════════════════════════════════════════════╣
║ • Trained on 32×32 images only (not suitable for hi-res)     ║
║ • 10 classes only (CIFAR-10 categories)                      ║
║ • Trained on subset (10K) for demo — production would use    ║
║   full dataset or larger model                               ║
║ • Accuracy may degrade on out-of-distribution images         ║
╚══════════════════════════════════════════════════════════════╝
"""
    return card

card = generate_model_card(profiles, "ONNX Runtime INT8")
print(card)

## 5.3 Per-Class Performance Analysis

Aggregate accuracy can hide class-level issues. A model with 80% overall accuracy might be 95% on "ship" but 50% on "cat". In production, this matters.

In [ ]:
# ──────────────────────────────────────────────
# Per-class accuracy analysis (ONNX INT8 model)
# ──────────────────────────────────────────────

def per_class_accuracy(session, test_loader, n_classes=10):
    """Compute per-class accuracy and confusion data."""
    input_name = session.get_inputs()[0].name
    class_correct = np.zeros(n_classes)
    class_total = np.zeros(n_classes)
    all_preds = []
    all_labels = []

    for images, labels in test_loader:
        outputs = session.run(None, {input_name: images.numpy()})
        preds = np.argmax(outputs[0], axis=1)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

        for pred, label in zip(preds, labels.numpy()):
            class_total[label] += 1
            if pred == label:
                class_correct[label] += 1

    class_acc = class_correct / class_total * 100
    return class_acc, np.array(all_preds), np.array(all_labels)


class_acc, all_preds, all_labels = per_class_accuracy(ort_quant_session, test_loader)

# Plot per-class accuracy
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(CLASSES)), class_acc,
              color=plt.cm.Set3(np.linspace(0, 1, 10)), edgecolor="gray")
ax.set_xticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Per-Class Accuracy (ONNX INT8 Model)", fontweight="bold")
ax.axhline(y=np.mean(class_acc), color="red", linestyle="--",
           label=f"Mean: {np.mean(class_acc):.1f}%")

for bar, acc in zip(bars, class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{acc:.1f}%", ha="center", fontsize=9)

ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(f"\nBest class:  {CLASSES[np.argmax(class_acc)]:>12s} ({np.max(class_acc):.1f}%)")
print(f"Worst class: {CLASSES[np.argmin(class_acc)]:>12s} ({np.min(class_acc):.1f}%)")

In [ ]:
# ──────────────────────────────────────────────
# Confusion Matrix
# ──────────────────────────────────────────────

def plot_confusion_matrix(preds, labels, class_names):
    """Plot a normalized confusion matrix."""
    n = len(class_names)
    cm = np.zeros((n, n), dtype=int)
    for pred, label in zip(preds, labels):
        cm[label][pred] += 1

    # Normalize by row (true class)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=100)

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion Matrix (%) — ONNX INT8 Model", fontweight="bold")

    # Annotate cells
    for i in range(n):
        for j in range(n):
            color = "white" if cm_norm[i, j] > 50 else "black"
            ax.text(j, i, f"{cm_norm[i, j]:.0f}",
                    ha="center", va="center", color=color, fontsize=9)

    plt.colorbar(im, ax=ax, label="% of true class")
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(all_preds, all_labels, CLASSES)

## 5.4 Comprehensive Project Report

Below is the complete summary following the structure: **Problem → Dataset → Approach → Results → Optimization → Next Steps**.

In [ ]:
# ──────────────────────────────────────────────
# Final summary: all file sizes
# ──────────────────────────────────────────────

print("═" * 60)
print("  SAVED MODEL ARTIFACTS")
print("═" * 60)

model_dir = Path("models")
for fpath in sorted(model_dir.glob("*")):
    size = get_file_size_mb(fpath)
    print(f"  {fpath.name:<35s} {size:>8.2f} MB")

print(f"\n{'─' * 60}")
total_size = sum(get_file_size_mb(f) for f in model_dir.glob("*"))
print(f"  {'Total':<35s} {total_size:>8.2f} MB")

In [ ]:
# ──────────────────────────────────────────────
# Final optimization summary visualization
# ──────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 7))

for i, (name, p) in enumerate(profiles.items()):
    ax.scatter(
        p["size_mb"], p["accuracy"],
        s=p["throughput_ips"] / 3,  # Size represents throughput
        color=colors[i], alpha=0.7, edgecolors="black", linewidth=0.5,
        label=f"{name} ({p['throughput_ips']:.0f} img/s)",
        zorder=3,
    )

ax.set_xlabel("Model Size (MB)", fontsize=12)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title("The Optimization Landscape\n(bubble size = throughput)",
             fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=8)
ax.grid(True, alpha=0.3)

# Ideal corner annotation
ax.annotate("← Ideal\n(small, accurate)",
            xy=(0.02, 0.98), xycoords="axes fraction",
            fontsize=9, color="gray", fontstyle="italic")

plt.tight_layout()
plt.show()

---

# Project Report: CIFAR-10 Model Optimization for Production

## Problem

Deep learning models trained for image classification are typically stored in FP32 format and run in PyTorch's eager execution mode. This is fine for research but creates challenges for production deployment: models are unnecessarily large, inference is slower than it needs to be, and deployment is locked to the Python/PyTorch ecosystem.

**Goal:** Take a trained CV model and optimize it for production deployment by reducing size, improving latency, and enabling cross-platform serving.

## Dataset

CIFAR-10: 60,000 32×32 color images across 10 classes. We used a 10K training subset for fast iteration and the full 10K test set for evaluation. The techniques demonstrated scale to any image classification model and dataset.

## Approach

We applied four complementary optimization techniques:

1. **Quantization** (Dynamic, Static, FP16) — Reduced numerical precision of weights and activations
2. **Pruning** — Removed up to 80% of weights with minimal accuracy loss
3. **Knowledge Distillation** — Trained a 10× smaller student model using the teacher's soft predictions
4. **ONNX Conversion** — Exported to ONNX for cross-platform deployment with graph-level optimizations

## Results

See the comprehensive comparison table and visualizations above. Key findings:

- **ONNX Runtime** consistently outperformed PyTorch inference on CPU due to graph optimizations
- **INT8 quantization** provided the best size reduction (~2–4×) with minimal accuracy loss (typically < 1%)
- **Knowledge distillation** successfully transferred knowledge to a 10× smaller model
- **Pruning** showed models tolerate 60–80% weight removal before significant degradation

## Optimization

The recommended production configuration is **ONNX Runtime with INT8 quantization** because it offers the best balance of size, speed, accuracy, and deployment flexibility.

## Next Steps

1. **Quantization-Aware Training (QAT):** Fine-tune with fake quantization nodes for even better INT8 accuracy
2. **TensorRT:** For NVIDIA GPU deployment, TensorRT can provide additional 2–5× speedup
3. **Model architecture search:** Use NAS or EfficientNet-style scaling to find optimal architecture
4. **Structured pruning + fine-tuning:** Remove entire filters and retrain for hardware-friendly speedups
5. **Serve with Triton Inference Server:** Add adaptive batching, model versioning, and A/B testing
6. **Edge deployment:** Export to TFLite (mobile) or Core ML (iOS) using ONNX as the intermediate format
7. **Monitoring:** Add data drift detection and accuracy monitoring in production

---

# 6. Summary and Key Takeaways <a id='6-summary-and-next-steps'></a>

### What We Covered

| Topic | Key Technique | Impact |
|---|---|---|
| Quantization | FP32 → INT8/FP16 | 2–4× smaller, faster inference |
| Pruning | Remove low-magnitude weights | 60–80% sparsity with < 2% accuracy loss |
| Knowledge Distillation | Teacher → Student | 10× fewer parameters, teacher-level accuracy |
| ONNX Conversion | PyTorch → ONNX Runtime | Cross-platform, graph-optimized inference |
| Serving Strategies | Batch vs. Real-time, Edge vs. Cloud | Choose based on latency, privacy, and cost |
| Inference Pipeline | Preprocessing + Model + Postprocessing | Production-ready, logged, and monitored |

### Rules of Thumb for Production ML

1. **Always profile before optimizing.** Measure what's actually slow.
2. **ONNX + INT8 quantization is the lowest-hanging fruit.** Do it first.
3. **Knowledge distillation is underrated.** It's free accuracy for smaller models.
4. **Batch size is a lever.** Tune it for your latency/throughput requirements.
5. **Model Cards are not optional.** Document everything for future you.
6. **Test on the target hardware.** CPU benchmarks don't predict GPU performance.

### Recommended Reading

- [ONNX Runtime Documentation](https://onnxruntime.ai/docs/)
- [PyTorch Quantization Guide](https://pytorch.org/docs/stable/quantization.html)
- [The Lottery Ticket Hypothesis (Frankle & Carlin, 2019)](https://arxiv.org/abs/1803.03635)
- [Distilling the Knowledge in a Neural Network (Hinton et al., 2015)](https://arxiv.org/abs/1503.02531)
- [MLOps: Machine Learning Operations (Google Cloud)](https://cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)